<a href="https://colab.research.google.com/github/thanhnam21122003/data-science-journey/blob/main/XAI_decision_tree_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import libraries**

In [ ]:
import pandas as pd
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, plot_tree

**Load and Encode the dataset**

In [ ]:
# Load the dataset
df = pd.read_csv(
    '/content/classification_ds_data.csv',
    names=['Điểm_Tốt_Nghiệp', 'Chứng_chỉ_Ielts', 'Cộng_Điểm_Dân_Tộc', 'Đậu_ĐH'],
    header=0
)
print("Original dataset:")
display(df)

In [ ]:
def encode_df(df):
# Encode categorical features and target
# We map 'Yes'/'No' in a way that ensures 'Yes' (0) goes to the left child, 'No' (1) to the right
    df2 = df.copy()
    bin_map = {'Có': 0, 'Không': 1}
    df2['Chứng_chỉ_Ielts']   = df2['Chứng_chỉ_Ielts'].map(bin_map)
    df2['Cộng_Điểm_Dân_Tộc'] = df2['Cộng_Điểm_Dân_Tộc'].map(bin_map)
    df2['Đậu_ĐH']            = df2['Đậu_ĐH'].map({'Có': 1, 'Không': 0})
    return df2

In [ ]:
# Encode
df_encoded = encode_df(df)
print("Encoded dataset:")
display(df_encoded)

**Compute Root Entropy of the target**

In [ ]:
def entropy(probs):
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

In [ ]:
root_probs = df_encoded['Đậu_ĐH'].value_counts(normalize=True).sort_index().values
root_entropy = entropy(root_probs)
print(f"Root entropy của Đậu_ĐH = {root_entropy:.4f}")

**Compute Information Gain for each feature**

In [ ]:
features = ['Chứng_chỉ_Ielts', 'Điểm_Tốt_Nghiệp', 'Cộng_Điểm_Dân_Tộc']
for f in features:
  vals = sorted(df_encoded[f].unique())
  thresholds = [0.5] if f in ['Chứng_chỉ_Ielts','Cộng_Điểm_Dân_Tộc'] else [(vals[i]+vals[i+1])/2 for i in range(len(vals)-1)]

In [ ]:
print("\nInformation Gain cho từng thuộc tính:")
features = ['Chứng_chỉ_Ielts', 'Điểm_Tốt_Nghiệp', 'Cộng_Điểm_Dân_Tộc']
for feat in features:
    print(f"\n- Thuộc tính: {feat}")
    vals = sorted(df_encoded[feat].unique())
    thresholds = [0.5] if feat in ['Chứng_chỉ_Ielts','Cộng_Điểm_Dân_Tộc'] \
                 else [(vals[i]+vals[i+1])/2 for i in range(len(vals)-1)]
    best_gain, best_thr = -1, None
    for thr in thresholds:
        left  = df_encoded[df_encoded[feat] <= thr]['Đậu_ĐH']
        right = df_encoded[df_encoded[feat] >  thr]['Đậu_ĐH']
        pL, pR = left.value_counts(normalize=True).values, right.value_counts(normalize=True).values
        eL, eR = entropy(pL), entropy(pR)
        wE     = (len(left)/len(df_encoded))*eL + (len(right)/len(df_encoded))*eR
        ig     = root_entropy - wE
        print(f"    Ngưỡng ≤ {thr}: Ent(L)={eL:.4f}, Ent(R)={eR:.4f}, Ent trọng số={wE:.4f}, IG={ig:.4f}")
        if ig > best_gain:
            best_gain, best_thr = ig, thr
    print(f"  → Chọn ngưỡng {best_thr} với IG = {best_gain:.4f}")

In [ ]:
X   = df_encoded[features]
y   = df_encoded['Đậu_ĐH']
clf = DecisionTreeClassifier(criterion='entropy', random_state=0)
clf.fit(X, y)

plt.figure(figsize=(10,6))
plot_tree(
    clf,
    feature_names=['Chứng chỉ IELTS','Điểm Tốt Nghiệp','Cộng Điểm Dân Tộc'],
    class_names=['Không','Có'],
    filled=True, rounded=True, proportion=True, impurity=False, fontsize=10
)
plt.title('Decision Tree (Entropy)')
plt.show()

In [ ]:
_mapping = {'Có': 0, 'Không': 1}

def encode_input(record):
    return [
        _mapping[record['Chứng_chỉ_Ielts']],
        record['Điểm_Tốt_Nghiệp'],
        _mapping[record['Cộng_Điểm_Dân_Tộc']]
    ]

def predict_pass(record):
    x_list = encode_input(record)
    df_new = pd.DataFrame([x_list], columns=features)
    pred   = clf.predict(df_new)[0]
    return 'Có' if pred == 1 else 'Không'

new_student = {
    'Điểm_Tốt_Nghiệp': 2.0,
    'Chứng_chỉ_Ielts': 'Có',
    'Cộng_Điểm_Dân_Tộc': 'Không'
}
print("Kết quả dự đoán:", predict_pass(new_student))

In [ ]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

def predict_and_plot(diem, ielts, dan_toc):
    try:
        record = {
            'Điểm_Tốt_Nghiệp': float(diem),
            'Chứng_chỉ_Ielts': ielts,
            'Cộng_Điểm_Dân_Tộc': dan_toc
        }
        result = predict_pass(record)
        fig, ax = plt.subplots(figsize=(8, 6))
        plot_tree(
            clf,
            feature_names=['Chứng chỉ IELTS','Điểm Tốt Nghiệp','Cộng Điểm Dân Tộc'],
            class_names=['Không','Có'],
            filled=True, rounded=True, proportion=True, impurity=False,
            fontsize=8, ax=ax
        )
        ax.set_title('Decision Tree (Entropy)')
        plt.tight_layout()
        return result, fig
    except Exception as e:
        return f"Lỗi nhập liệu: {e}", None

iface = gr.Interface(
    fn=predict_and_plot,
    inputs=[
        gr.Number(
            label="Điểm Tốt Nghiệp",
            value=None,
            placeholder="Ví dụ: 17.0"
        ),
        gr.Radio(
            choices=['Có', 'Không'],
            label="Chứng chỉ IELTS",
            value='Có'
        ),
        gr.Radio(
            choices=['Có', 'Không'],
            label="Cộng Điểm Dân Tộc",
            value='Không'
        ),
    ],
    outputs=[
        gr.Textbox(label="Dự đoán Đậu Đại Học"),
        gr.Plot(label="Decision Tree")
    ],
    title="Demo Student Pass Predictor",
    description="""
**Hướng dẫn nhập liệu**
- **Điểm Tốt Nghiệp**: Nhập số điểm tốt nghiệp (float).
- **Chứng chỉ IELTS**: Chọn `Có` nếu bạn đã có chứng chỉ IELTS, ngược lại chọn `Không`.
- **Cộng Điểm Dân Tộc**: Chọn `Có` nếu bạn được cộng điểm dân tộc, ngược lại chọn `Không`.
Bấm **Run** để xem kết quả dự đoán và cây quyết định.
"""
)

iface.launch()